In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

In [3]:
plt.rcParams['font.sans-serif'] = ['SimHei']    # Windows自带黑体
plt.rcParams['axes.unicode_minus'] = False      # 解决负号显示问题

In [8]:
df = pd.read_excel('diabetic_data.xlsx')

In [5]:
# 2.1.1 年龄 vs 并发症发生率
# =================================================================
age_labels = {
    1: '0-10岁', 2: '10-20岁', 3: '20-30岁',
    4: '30-40岁', 5: '40-50岁', 6: '50-60岁',
    7: '60-70岁', 8: '70-80岁', 9: '80-90岁', 10: '90-100岁'
}

age_comp = df.groupby('年龄')['has_complication'].mean() * 100

plt.figure(figsize=(10, 5))
x = [age_labels[i] for i in age_comp.index]
y = age_comp.values
plt.plot(x, y, marker='o', linewidth=2.5, markersize=8, color='#E74C3C')
plt.bar(x, y, alpha=0.3, color='#3498DB')
plt.title('各年龄段糖尿病并发症发生率', fontsize=14)
plt.ylabel('并发症发生率 (%)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('年龄_并发症率.png', dpi=300)
plt.close()

print("=== 各年龄段并发症发生率（%）===")
for age, rate in age_comp.items():
    print(f"{age_labels[age]}: {rate:.1f}%")

=== 各年龄段并发症发生率（%）===
0-10岁: 8.7%
10-20岁: 4.2%
20-30岁: 14.9%
30-40岁: 21.6%
40-50岁: 20.8%
50-60岁: 21.5%
60-70岁: 22.0%
70-80岁: 20.7%
80-90岁: 18.1%
90-100岁: 15.9%


In [6]:
# 2.1.2 药物数量 vs 并发症
# =================================================================
drug_comp = df.groupby('使用药物数量')['has_complication'].agg(['mean', 'count'])
drug_comp['mean'] = drug_comp['mean'] * 100

plt.figure(figsize=(10, 5))
plt.bar(drug_comp.index, drug_comp['mean'], color='#2ECC71', alpha=0.7)
plt.plot(drug_comp.index, drug_comp['mean'], marker='o', color='#E67E22', linewidth=2)
plt.title('使用药物数量与并发症发生率', fontsize=14)
plt.xlabel('使用药物数量', fontsize=12)
plt.ylabel('并发症发生率 (%)', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('药物数量_并发症.png', dpi=300)
plt.close()

print("\n=== 药物数量与并发症发生率（%）===")
print(drug_comp.round(1))


=== 药物数量与并发症发生率（%）===
        mean  count
使用药物数量             
1       14.9    262
2       13.8    470
3       12.8    900
4       13.1   1417
5       13.6   2017
...      ...    ...
72       0.0      3
74       0.0      1
75       0.0      2
79       0.0      1
81       0.0      1

[75 rows x 2 columns]


In [9]:
# 2.2.1 特征相关性热力图
# =================================================================
corr_features = [
    '性别', '年龄', '住院天数', '使用药物数量',
    '最高血清血糖', '糖化血红蛋白结果',
    '门诊就诊次数', '急诊就诊次数', '住院次数'
]
corr = df[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('特征相关性热力图', fontsize=14)
plt.tight_layout()
plt.savefig('相关性热力图.png', dpi=300)
plt.close()

print("\n=== 两个血糖指标相关系数 ===")
print(corr.loc['糖化血红蛋白结果', '最高血清血糖'].round(3))


=== 两个血糖指标相关系数 ===
-0.064


In [10]:
# 2.2.2 高风险人群对比
# =================================================================
df['高风险人群'] = (
    (df['性别'] == 1) &          
    (df['糖化血红蛋白结果'] == 2) &  
    (df['住院天数'] > 7)          
)

high_risk_rate = df[df['高风险人群']==True]['has_complication'].mean()*100
normal_rate = df[df['高风险人群']==False]['has_complication'].mean()*100

plt.figure(figsize=(6, 5))
plt.bar(['普通人群', '高风险人群'], [normal_rate, high_risk_rate], color=['#3498DB', '#E74C3C'])
plt.title('高风险人群 vs 普通人群 并发症率', fontsize=14)
plt.ylabel('并发症发生率 (%)', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('高风险人群对比.png', dpi=300)
plt.close()

print("\n=== 高风险人群并发症率对比 ===")
print(f"普通人群并发症率: {normal_rate:.1f}%")
print(f"高风险人群并发症率: {high_risk_rate:.1f}%")
print(f"风险提升: {high_risk_rate/normal_rate - 1:.1%}")


=== 高风险人群并发症率对比 ===
普通人群并发症率: 20.3%
高风险人群并发症率: 34.1%
风险提升: 68.4%


In [11]:
# 2.3.1 风险因素重要性排序
# =================================================================
X = df[corr_features]
y = df['has_complication']
mi = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
mi_series.plot(kind='barh', color='#9B59B6')
plt.title('并发症风险因素重要性（互信息值）', fontsize=14)
plt.xlabel('互信息（越大越重要）', fontsize=12)
plt.tight_layout()
plt.savefig('风险因素排序.png', dpi=300)
plt.close()

print("\n=== 风险因素重要性排序 ===")
print(mi_series.sort_values(ascending=False).round(3))


=== 风险因素重要性排序 ===
性别          0.011
住院次数        0.009
年龄          0.007
住院天数        0.005
门诊就诊次数      0.003
糖化血红蛋白结果    0.003
使用药物数量      0.002
最高血清血糖      0.000
急诊就诊次数      0.000
dtype: float64


In [12]:
# 2.3.2 高风险人群雷达图
# =================================================================
features = ['年龄', '糖化血红蛋白', '血糖', '住院天数', '用药数', '急诊次数']
high_risk = df[df['高风险人群']==True]
normal = df[df['高风险人群']==False]

def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-8)

high_vals = [
    normalize(high_risk['年龄']).mean(),
    normalize(high_risk['糖化血红蛋白结果']).mean(),
    normalize(high_risk['最高血清血糖']).mean(),
    normalize(high_risk['住院天数']).mean(),
    normalize(high_risk['使用药物数量']).mean(),
    normalize(high_risk['急诊就诊次数']).mean()
]

normal_vals = [
    normalize(normal['年龄']).mean(),
    normalize(normal['糖化血红蛋白结果']).mean(),
    normalize(normal['最高血清血糖']).mean(),
    normalize(normal['住院天数']).mean(),
    normalize(normal['使用药物数量']).mean(),
    normalize(normal['急诊就诊次数']).mean()
]

angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
high_vals += high_vals[:1]
normal_vals += normal_vals[:1]
angles += angles[:1]

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)
ax.plot(angles, high_vals, 'o-', linewidth=2, label='高风险人群', color='#E74C3C')
ax.fill(angles, high_vals, alpha=0.25, color='#E74C3C')
ax.plot(angles, normal_vals, 'o-', linewidth=2, label='普通人群', color='#3498DB')
ax.fill(angles, normal_vals, alpha=0.25, color='#3498DB')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(features)
plt.legend(loc='upper right')
plt.title('糖尿病高风险人群画像雷达图', fontsize=14)
plt.tight_layout()
plt.savefig('高风险人群雷达图.png', dpi=300)
plt.close()